# Sumativa 2 · Taller 1 · Big Data (UNAB)
## Análisis histórico de los Juegos Olímpicos con Apache Spark

**Entregable:** notebook ejecutable (`.ipynb`) para **VS Code + Jupyter**. **Tecnologías:** Python, PySpark (RDDs, DataFrames, Spark SQL).

**Objetivo:** integrar cinco entidades (deportistas, equipos, resultados, eventos y juegos), distribuir el procesamiento en Spark, transformar registros y calcular los agregados solicitados en la rúbrica.

**Archivos proporcionados:** `deportista.csv`, `deportista2.csv`, `equipo.csv`, `evento.csv`, `resultados.csv` y `juegos.json`. Se utilizan **todos** y se mantienen intactos en `data/`.

**Tratamiento transparente de datos:** `deportista.csv` incluye una coma sobrante en el registro ID 16587, y ambos archivos tienen pesos decimales. `resultados.csv` tiene nueve `evento_id = #N/A`; 200 resultados se asocian a deportistas con `equipo_id = 0` (sin equipo en el catálogo). Se mantienen los resultados mediante uniones `left`, incluso si falta un nombre de evento o equipo. Las ausencias se explican, no se inventan valores.

La pauta contiene ocho secciones técnicas, una presentación grupal y requisitos formales. Este notebook cubre las partes de código; el grupo debe realizar su propia grabación de video (7–10 minutos) y entregar el ZIP final en Canvas.

## 1. Configuración inicial del entorno 

Instalar **Java JDK 17 o superior**, Python y PySpark en el entorno local. En la terminal de VS Code, con el entorno virtual activado:

```powershell
python -m pip install -r requirements.txt
java -version
```

Si `java -version` falla, instalar un JDK y configurar `JAVA_HOME`/`PATH`. El código siguiente verifica las dependencias, configura variables y crea **SparkSession y SparkContext**. No hace falta Google Colab.

In [1]:
from pathlib import Path
import os
import sys
import shutil
import csv
from src.olimpicos import (campos_deportista, parsear_deportista, entero_o_none, decimal_o_none,
    preparar_eventos, calcular_columnas, medallas_equipo, estadisticas)

BASE = Path.cwd().resolve()
DATA = BASE / "data"
GENERADOS = BASE / "data_procesada"
GENERADOS.mkdir(exist_ok=True)

if not DATA.is_dir():
    raise FileNotFoundError(f"No existe {DATA}. Abre Jupyter desde la raíz del proyecto")
if shutil.which("java") is None:
    raise RuntimeError("Java no está en PATH. Instala JDK 17 o 21, configura JAVA_HOME si hace falta, y reinicia VS Code")

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

try:
    import pyspark
    from pyspark.sql import SparkSession, functions as F
    from pyspark.sql.functions import broadcast
    from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType
except ImportError as exc:
    raise ImportError("Falta PySpark. En la terminal: python -m pip install -r requirements.txt") from exc

print("Python:", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("Java disponible:", shutil.which("java"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "detectado a través del PATH"))
print("Ruta de datos:", DATA)

Python: 3.12.6
PySpark: 4.0.3
Java disponible: C:\Users\crist\AppData\Local\Programs\Eclipse Adoptium\jdk-21.0.13.9-hotspot\bin\java.EXE
JAVA_HOME: detectado a través del PATH
Ruta de datos: C:\Users\crist\Desktop\Magister\MCDI502\olympic-games-bigdata-analytics\data


In [2]:
spark = (SparkSession.builder
         .appName("UNAB_Sumativa2_JuegosOlimpicos")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "5")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("SparkSession creada | Spark:", spark.version)
print("SparkContext activo:", sc.appName)
print("Máquina local, paralelismo por defecto:", sc.defaultParallelism)

SparkSession creada | Spark: 4.0.3
SparkContext activo: UNAB_Sumativa2_JuegosOlimpicos
Máquina local, paralelismo por defecto: 8


### Control de archivos originales

La evaluación exige ambas fuentes de deportistas y las cuatro fuentes adicionales. El programa confirma que **todos los archivos están presentes**, para evitar resultados parciales sin advertencia.

In [3]:
archivos_esperados = [
    "deportista.csv", "deportista2.csv", "resultados.csv",
    "equipo.csv", "evento.csv", "juegos.json"
]
for nombre in archivos_esperados:
    ruta = DATA / nombre
    print(("OK       " if ruta.is_file() else "FALTANTE ") + nombre)

faltantes = [n for n in archivos_esperados if not (DATA / n).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan datos de la asignatura: {faltantes}")
print("Archivos requeridos: 6 de 6.")

OK       deportista.csv
OK       deportista2.csv
OK       resultados.csv
OK       equipo.csv
OK       evento.csv
OK       juegos.json
Archivos requeridos: 6 de 6.


## 2. RDDs: creación, unión y conversión a DataFrame 
Se crea el RDD **`deportista` con seis particiones** a partir de `deportista.csv`, el RDD `deportista2` desde su segundo archivo y su unión **`deportistaTotal`**. Se cuentan los registros y se convierte la unión en el DataFrame `deportista` con las siete columnas solicitadas.

Las dos fuentes originales vienen **sin encabezados**. Solo una fila termina con una coma extra (ID 16587); se corrige al parsear, sin eliminarla. Los pesos permiten decimales y por eso la columna `peso` se representa con `DoubleType`.

In [4]:
deportista = sc.textFile(str(DATA / "deportista.csv")).repartition(6)
deportista2 = sc.textFile(str(DATA / "deportista2.csv"))
deportistaTotal = deportista.union(deportista2)

cuenta_deportista = deportista.count()
cuenta_deportista2 = deportista2.count()
cuenta_total = deportistaTotal.count()
print("RDD deportista: particiones:", deportista.getNumPartitions())
print("RDD deportista: registros:", cuenta_deportista)
print("RDD deportista2: registros:", cuenta_deportista2)
print("RDD deportistaTotal: registros:", cuenta_total)
print("Ejemplo de fila:", deportistaTotal.take(1))

assert deportista.getNumPartitions() == 6
assert cuenta_deportista == 67786, "El primer archivo no conserva sus 67.786 registros"
assert cuenta_deportista2 == 67785, "El segundo archivo no conserva sus 67.785 registros"
assert cuenta_total == 135571, "Se esperaba la unión de los 135.571 registros"
RDD_deportista = deportista  # guardamos el RDD antes de asignar el DataFrame con el nombre requerido

RDD deportista: particiones: 6
RDD deportista: registros: 67786
RDD deportista2: registros: 67785
RDD deportistaTotal: registros: 135571
Ejemplo de fila: ['31,Evald rma rman ,1,24,174,70,331']


In [5]:
esquema_deportistas = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("genero", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("altura", IntegerType(), True),  # centímetros
    StructField("peso", DoubleType(), True),    # kg, conserva decimales
    StructField("equipo_id", IntegerType(), True),
])

deportista = spark.createDataFrame(deportistaTotal.map(parsear_deportista), schema=esquema_deportistas)
print("DataFrame deportista creado:")
deportista.printSchema()
deportista.show(6, truncate=False)
print("Registros del DataFrame deportista:", deportista.count())
assert deportista.count() == cuenta_total, "Se perdieron filas durante la conversión a DataFrame"

DataFrame deportista creado:
root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: integer (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

+-------------+----------------------------+------+----+------+----+---------+
|deportista_id|nombre                      |genero|edad|altura|peso|equipo_id|
+-------------+----------------------------+------+----+------+----+---------+
|31           |Evald rma rman              |1     |24  |174   |70.0|331      |
|32           |Olav Augunson Aarnes        |1     |23  |0     |0.0 |742      |
|33           |Mika Lauri Aarnikka         |1     |24  |187   |76.0|350      |
|34           |Jamale Djamel Aarrass Ahrass|1     |30  |187   |76.0|362      |
|35           |Dagfinn Sverre Aarskog      |1     |24  |190   |98.0|742      |
|36           |Stefan Remco Aartsen        |1     |21  

## 3. RDDs: transformaciones 

- `MayorEdad`: deportistas con edad **mayor o igual a 18**.
- `Deportistas_mujer`: registros con **género = 2**.
- `deportistaTotal_mayusculas`: todos los caracteres alfabéticos del RDD original en mayúsculas.

Se muestran cantidades y ejemplos para evidenciar los filtros y el resultado de la transformación.

In [6]:
MayorEdad = deportistaTotal.filter(lambda fila: (entero_o_none(campos_deportista(fila)[3]) or 0) >= 18)
Deportistas_mujer = deportistaTotal.filter(lambda fila: entero_o_none(campos_deportista(fila)[2]) == 2)
deportistaTotal_mayusculas = deportistaTotal.map(lambda fila: fila.upper())

print("RDD MayorEdad (edad >= 18):", MayorEdad.count(), "registros")
print("RDD Deportistas_mujer (sexo = 2):", Deportistas_mujer.count(), "registros")
print("Ejemplo MayorEdad:", MayorEdad.take(3))
print("Ejemplo Deportistas_mujer:", Deportistas_mujer.take(3))
print("RDD original:", deportistaTotal.take(2))
print("RDD en mayúsculas:", deportistaTotal_mayusculas.take(2))

RDD MayorEdad (edad >= 18): 123244 registros
RDD Deportistas_mujer (sexo = 2): 33981 registros
Ejemplo MayorEdad: ['31,Evald rma rman ,1,24,174,70,331', '32,Olav Augunson Aarnes,1,23,0,0,742', '33,Mika Lauri Aarnikka,1,24,187,76,350']
Ejemplo Deportistas_mujer: ['37,Ann Kristin Aarnes,2,23,182,64,742', '95,Georgia Abatzidou,2,35,155,43,419', '151,Mary Perkins Ives Abbott Perkins ,2,42,0,0,1096']
RDD original: ['31,Evald rma rman ,1,24,174,70,331', '32,Olav Augunson Aarnes,1,23,0,0,742']
RDD en mayúsculas: ['31,EVALD RMA RMAN ,1,24,174,70,331', '32,OLAV AUGUNSON AARNES,1,23,0,0,742']


## 4. DataFrames: creación, limpieza, optimización e integración 

Se crean los DataFrames `Evento`, `Resultado`, `Equipos` y `Juego` y se integran con `deportista` mediante las claves del diagrama de la guía.

**Particularidades de los archivos reales:**

- `resultados.csv` está separado por punto y coma (`;`); las medallas son `Gold`, `Silver`, `Bronze` o `NA`.
- `evento.csv` tiene 280 filas completas envueltas en comillas. Se guarda una copia normalizada en `data_procesada/` (original intacto) y se confirma el total de 765 eventos.
- `juegos.json` contiene el **año** en el campo llamado `temporada`, y **Verano/Invierno** en el campo llamado `ciudad`; se renombran a `anio` y `temporada` para que tengan significado estadístico.
- Nueve resultados carecen de `evento_id` válido (`#N/A`); en 200 participaciones el `equipo_id` es 0 y no hay equipo en el catálogo. Se convierten IDs ausentes a `NULL` mediante `try_cast` y se usan `left joins` para **preservar los 271.116 resultados**; los valores desconocidos quedan como nulos. No se imputa información.
- Se optimizan uniones con `broadcast()` para dimensiones pequeñas y después se utiliza `persist()` para evitar recomputaciones costosas.

In [7]:
ruta_evento_limpio = GENERADOS / "evento_limpio.csv"
num_eventos, filas_corregidas = preparar_eventos(DATA / "evento.csv", ruta_evento_limpio)
print(f"Eventos preservados: {num_eventos}; líneas normalizadas: {filas_corregidas}")
assert num_eventos == 765, "Cambió la cantidad esperada de eventos; revisar fuente antes de continuar"

Eventos preservados: 765; líneas normalizadas: 280


In [8]:
Evento = (spark.read.option("header", True).option("inferSchema", True)
          .csv(str(ruta_evento_limpio))
          .withColumnRenamed("cod_evento", "evento_id")
          .withColumnRenamed("evento", "nombre_evento"))

Resultado = (spark.read.option("header", True).option("inferSchema", True)
             .option("sep", ";").csv(str(DATA / "resultados.csv"))
             .withColumn("evento_id", F.expr("try_cast(evento_id AS INT)")))

Equipos = (spark.read.option("header", True).option("inferSchema", True)
           .csv(str(DATA / "equipo.csv"))
           .withColumnRenamed("id", "equipo_id")
           .withColumnRenamed("equipo", "nombre_equipo"))

Juego = (spark.read.option("multiLine", True).json(str(DATA / "juegos.json"))
         .withColumnRenamed("temporada", "anio")
         .withColumnRenamed("ciudad", "temporada")
         .withColumnRenamed("ano", "edicion"))

for nombre, df in [("Evento", Evento), ("Resultado", Resultado), ("Equipos", Equipos), ("Juego", Juego)]:
    print(f"{nombre}: {df.count()} filas | columnas: {df.columns}")
    df.show(3, truncate=False)

assert Evento.count() == 765, "La lectura de los eventos está incompleta"
assert Resultado.count() == 271116, "La lectura de resultados debe conservar las 271.116 filas"
assert Equipos.count() == 1184 and Juego.count() == 51
print("Resultados con evento_id desconocido:", Resultado.filter(F.col("evento_id").isNull()).count())

Evento: 765 filas | columnas: ['evento_id', 'nombre_evento', 'deporte_id']
+---------+----------------------------+----------+
|evento_id|nombre_evento               |deporte_id|
+---------+----------------------------+----------+
|1        |Basketball Men's Basketball |1         |
|2        |Judo Men's Extra-Lightweight|2         |
|3        |Football Men's Football     |3         |
+---------+----------------------------+----------+
only showing top 3 rows
Resultado: 271116 filas | columnas: ['resultado_id', 'medalla', 'deportista_id', 'juego_id', 'evento_id']
+------------+-------+-------------+--------+---------+
|resultado_id|medalla|deportista_id|juego_id|evento_id|
+------------+-------+-------------+--------+---------+
|1           |NA     |1            |39      |1        |
|2           |NA     |2            |49      |2        |
|3           |NA     |3            |7       |3        |
+------------+-------+-------------+--------+---------+
only showing top 3 rows
Equipos: 1184 f

In [9]:
# Modelo relacional: resultado -> deportista -> equipo; resultado -> juego; resultado -> evento.
# Resultado es la tabla de hechos. Las uniones LEFT conservan resultados con referencias desconocidas.
df_integrado = (Resultado
    .join(deportista, on="deportista_id", how="left")
    .join(broadcast(Equipos), on="equipo_id", how="left")
    .join(broadcast(Juego), on="juego_id", how="left")
    .join(broadcast(Evento), on="evento_id", how="left")
    .withColumn("sexo", F.col("genero"))
)

resultados_sin_deportista = df_integrado.filter(F.col("nombre").isNull()).count()
resultados_sin_equipo = df_integrado.filter(F.col("nombre_equipo").isNull()).count()
resultados_sin_evento = df_integrado.filter(F.col("nombre_evento").isNull()).count()

print("Resultados originales:", Resultado.count())
print("DataFrame integrado:", df_integrado.count())
print("Sin deportista relacionado:", resultados_sin_deportista)
print("Sin equipo catalogado:", resultados_sin_equipo)
print("Sin nombre de evento:", resultados_sin_evento)
df_integrado.select("deportista_id", "nombre", "nombre_equipo", "medalla", "temporada", "anio", "nombre_evento").show(8, truncate=35)

assert resultados_sin_deportista == 0, "Hay deportistas no reconocidos: revisar las fuentes"
assert df_integrado.count() == Resultado.count(), "Las uniones alteraron el número de resultados"

Resultados originales: 271116
DataFrame integrado: 271116
Sin deportista relacionado: 0
Sin equipo catalogado: 200
Sin nombre de evento: 9
+-------------+------------------------+--------------+-------+---------+----+---------------------------------+
|deportista_id|                  nombre| nombre_equipo|medalla|temporada|anio|                    nombre_evento|
+-------------+------------------------+--------------+-------+---------+----+---------------------------------+
|            3|     Gunnar Nielsen Aaby|       Denmark|     NA|   Verano|1920|          Football Men's Football|
|            5|Christine Jacoba Aaftink|   Netherlands|     NA| Invierno|1988| Speed Skating Women's 500 metres|
|            5|Christine Jacoba Aaftink|   Netherlands|     NA| Invierno|1988|Speed Skating Women's 1000 metres|
|            5|Christine Jacoba Aaftink|   Netherlands|     NA| Invierno|1992| Speed Skating Women's 500 metres|
|            5|Christine Jacoba Aaftink|   Netherlands|     NA| Invier

## 5. Paralelismo: DataFrame en cinco particiones 

Se aplica `repartition(5)` y se evidencia el número de particiones con `rdd.getNumPartitions()`. El DataFrame se almacena en caché para reutilizarlo en los cálculos posteriores.

In [10]:
from pyspark import StorageLevel

df_paralelo = df_integrado.repartition(5).persist(StorageLevel.MEMORY_AND_DISK)
numero_filas_integradas = df_paralelo.count()  # materializa caché
particiones = df_paralelo.rdd.getNumPartitions()
print("Filas del DataFrame reparticionado:", numero_filas_integradas)
print("Número de particiones obtenido:", particiones)
assert particiones == 5, "Se esperaban exactamente 5 particiones"

Filas del DataFrame reparticionado: 271116
Número de particiones obtenido: 5


## 6. Inspección del DataFrame 

Evidencias requeridas por la pauta: **cantidad de filas**, **tipos de datos** y **esquema** del DataFrame integrado.

In [11]:
print("CANTIDAD DE FILAS:", df_paralelo.count())
print("TIPOS DE DATOS (dtypes):")
for nombre, tipo in df_paralelo.dtypes:
    print(f"  {nombre:22s} {tipo}")
print("ESQUEMA:")
df_paralelo.printSchema()
df_paralelo.show(5, truncate=38)

CANTIDAD DE FILAS: 271116
TIPOS DE DATOS (dtypes):
  evento_id              int
  juego_id               int
  equipo_id              int
  deportista_id          int
  resultado_id           int
  medalla                string
  nombre                 string
  genero                 int
  edad                   int
  altura                 int
  peso                   double
  nombre_equipo          string
  sigla                  string
  edicion                string
  temporada              string
  anio                   bigint
  nombre_evento          string
  deporte_id             string
  sexo                   int
ESQUEMA:
root
 |-- evento_id: integer (nullable = true)
 |-- juego_id: integer (nullable = true)
 |-- equipo_id: integer (nullable = true)
 |-- deportista_id: integer (nullable = true)
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nu

## 7. Columnas calculadas: IMC y Descripción_sexo 

- **IMC** = peso (kg) / [altura (metros)]². El archivo original usa **altura en centímetros**, por lo que se divide entre 100 antes del cálculo.
- Los valores **0** de peso/altura representan ausencia de información; se dejan como **NULL** para no crear IMCs incorrectos ni dividir por cero.
- **Descripción_sexo**: `1 = hombre`; `2 = mujer`.

In [12]:
df_calculado = calcular_columnas(df_paralelo)
print("Columnas calculadas disponibles:", "IMC" in df_calculado.columns, "Descripción_sexo" in df_calculado.columns)
df_calculado.select("deportista_id", "nombre", "altura", "peso", "IMC", "sexo", "Descripción_sexo").show(12, truncate=False)
assert {"IMC", "Descripción_sexo"}.issubset(set(df_calculado.columns))

Columnas calculadas disponibles: True True
+-------------+--------------------------------------+------+----+-----+----+----------------+
|deportista_id|nombre                                |altura|peso|IMC  |sexo|Descripción_sexo|
+-------------+--------------------------------------+------+----+-----+----+----------------+
|37210        |Jan Friesinger                        |179   |78.0|24.34|1   |hombre          |
|88177        |Sabahattin Olago                      |174   |70.0|23.12|1   |hombre          |
|23642        |Alistair Ian Cragg                    |180   |60.0|18.52|1   |hombre          |
|61765        |Jonathan Koch                         |183   |73.0|21.8 |1   |hombre          |
|114342       |Irina Vasilyevna Stankina             |164   |50.0|18.59|2   |mujer           |
|100302       |Anfisa Anatolyevna Reztsova Romanova  |162   |55.0|20.96|2   |mujer           |
|89554        |Policarpio Ortega                     |0     |0.0 |NULL |1   |hombre          |
|90648 

## 8. Agregaciones requeridas 

### 8.1 Medallas de oro, plata y bronce por equipo

Se cuentan registros de resultados por equipo con `Gold`, `Silver` y `Bronze`, excluyendo `NA`. En deportes colectivos puede haber varios registros (uno por deportista) para un mismo podio: **aquí contamos registros de medalla**, porque es lo que entrega la tabla Resultado. Los deportistas con equipo_id 0 no obtuvieron medallas en los datos recibidos.

In [13]:
medallas_por_equipo = medallas_equipo(df_calculado)
print("Conteo de medallas por equipo (registros con medalla):")
medallas_por_equipo.show(20, truncate=False)

Conteo de medallas por equipo (registros con medalla):
+---------+-------------+-----+----+-----+------+-----+
|equipo_id|nombre_equipo|sigla|oro |plata|bronce|total|
+---------+-------------+-----+----+-----+------+-----+
|1096     |United States|USA  |2515|1546 |1249  |5310 |
|978      |Soviet Union |URS  |1110|766  |728   |2604 |
|399      |Germany      |GER  |636 |603  |651   |1890 |
|413      |Great Britain|GBR  |531 |591  |578   |1700 |
|362      |France       |FRA  |452 |524  |582   |1558 |
|507      |Italy        |ITA  |536 |507  |487   |1530 |
|1012     |Sweden       |SWE  |450 |479  |510   |1439 |
|66       |Australia    |AUS  |343 |453  |506   |1302 |
|174      |Canada       |CAN  |422 |407  |407   |1236 |
|477      |Hungary      |HUN  |429 |329  |363   |1121 |
|302      |East Germany |GDR  |409 |325  |283   |1017 |
|705      |Netherlands  |NED  |275 |323  |395   |993  |
|874      |Russia       |RUS  |309 |291  |361   |961  |
|199      |China        |CHN  |323 |332  |273   |

### 8.2 Estadísticos de edad por tipo de medalla

La edad **0** se trata como dato desconocido y se excluye de los estadísticos. Se calculan **suma, promedio, máximo y mínimo** por tipo de medalla real.

In [14]:
edad_por_medalla = estadisticas(
    df_calculado.filter(F.col("medalla").isin("Gold", "Silver", "Bronze")), "edad", "medalla")
print("Estadísticos de edad según medalla:")
edad_por_medalla.show(truncate=False)

Estadísticos de edad según medalla:
+-------+---------+--------------------+-------------+-----------+-----------+
|medalla|suma_edad|cantidad_deportistas|promedio_edad|maximo_edad|minimo_edad|
+-------+---------+--------------------+-------------+-----------+-----------+
|Bronze |308071   |13006               |23.69        |72         |10         |
|Gold   |313374   |13224               |23.7         |64         |11         |
|Silver |304518   |12821               |23.75        |73         |11         |
+-------+---------+--------------------+-------------+-----------+-----------+



### 8.3 DataFrame `temporada`: estadísticos de altura por temporada

Para esta agregación se utiliza la variable **anio**, correspondiente al campo temporada del archivo original `juegos.json`. Se mantiene esta agrupación para respetar la estructura y denominación de los datos proporcionados en la evaluación. El renombramiento se realizó únicamente para mejorar la interpretación de las variables, diferenciando el año de celebración de la estación olímpica (Verano o Invierno).

Por lo tanto, para cada año, se calculan suma, promedio, máximo y mínimo de las alturas en **cm**. Los valores 0 se excluyen por ser datos ausentes.

In [15]:
temporada = estadisticas(df_calculado, "altura", "anio", nombre="altura_cm")
print("DataFrame temporada: altura agrupada por temporada olímpica:")
temporada.show(truncate=False)

DataFrame temporada: altura agrupada por temporada olímpica:
+----+--------------+--------------------+------------------+----------------+----------------+
|anio|suma_altura_cm|cantidad_deportistas|promedio_altura_cm|maximo_altura_cm|minimo_altura_cm|
+----+--------------+--------------------+------------------+----------------+----------------+
|1896|7946          |46                  |172.74            |188             |154             |
|1900|20490         |116                 |176.64            |191             |153             |
|1904|37443         |213                 |175.79            |195             |155             |
|1906|45799         |257                 |178.21            |196             |165             |
|1908|84333         |475                 |177.54            |201             |157             |
|1912|127940        |721                 |177.45            |200             |157             |
|1920|134802        |767                 |175.75            |197           

### 8.4 DataFrame `sexo`: estadísticos de edad por sexo

Se calculan suma, promedio, máximo y mínimo de edad agrupados por sexo. El sexo se presenta como **hombre / mujer** para facilitar la lectura.

In [16]:
sexo = estadisticas(
    df_calculado.filter(F.col("Descripción_sexo").isin("hombre", "mujer")),
    "edad", "Descripción_sexo")
print("DataFrame sexo: edad agrupada por sexo:")
sexo.show(truncate=False)

DataFrame sexo: edad agrupada por sexo:
+----------------+---------+--------------------+-------------+-----------+-----------+
|Descripción_sexo|suma_edad|cantidad_deportistas|promedio_edad|maximo_edad|minimo_edad|
+----------------+---------+--------------------+-------------+-----------+-----------+
|hombre          |4581905  |187544              |24.43        |97         |10         |
|mujer           |1619623  |74098               |21.86        |74         |11         |
+----------------+---------+--------------------+-------------+-----------+-----------+



## Complemento: consulta con Spark SQL

El indicador **ID 3.3** menciona consultas con Spark SQL. Para evidenciarlo se crea una vista temporal y se ejecuta una consulta SQL equivalente a una de las agrupaciones, sin reemplazar los cálculos anteriores.

In [17]:
df_calculado.createOrReplaceTempView("resultados_olimpicos")
consulta_sql = spark.sql("""
    SELECT anio,
           medalla,
           COUNT(*) AS registros_medalla
    FROM resultados_olimpicos
    WHERE medalla IN ('Gold', 'Silver', 'Bronze')
    GROUP BY anio, medalla
    ORDER BY anio, registros_medalla DESC
""")
print("Consulta Spark SQL: medallas por temporada:")
consulta_sql.show(truncate=False)

Consulta Spark SQL: medallas por temporada:
+----+-------+-----------------+
|anio|medalla|registros_medalla|
+----+-------+-----------------+
|1896|Gold   |62               |
|1896|Silver |43               |
|1896|Bronze |38               |
|1900|Silver |228              |
|1900|Gold   |201              |
|1900|Bronze |175              |
|1904|Gold   |173              |
|1904|Silver |163              |
|1904|Bronze |150              |
|1906|Gold   |157              |
|1906|Silver |156              |
|1906|Bronze |145              |
|1908|Gold   |294              |
|1908|Silver |281              |
|1908|Bronze |256              |
|1912|Gold   |326              |
|1912|Silver |315              |
|1912|Bronze |300              |
|1920|Gold   |493              |
|1920|Silver |448              |
+----+-------+-----------------+
only showing top 20 rows


## Verificaciones finales, cobertura y reproducibilidad

Esta sección comprueba los resultados estructurales y señala explícitamente cuándo faltan deportistas. Los `assert` verifican exigencias técnicas concretas de la pauta; **no equivalen a una calificación docente**.

In [18]:
assert RDD_deportista.getNumPartitions() == 6
assert deportista.count() == 135571, "Faltan deportistas"
assert df_paralelo.rdd.getNumPartitions() == 5
assert num_eventos == Evento.count() == 765
assert df_calculado.count() == 271116, "Se perdieron filas de resultados"
assert len(medallas_por_equipo.columns) >= 7
assert {"IMC", "Descripción_sexo"}.issubset(set(df_calculado.columns))
assert {"suma_edad", "promedio_edad", "maximo_edad", "minimo_edad"}.issubset(sexo.columns)
assert {"suma_altura_cm", "promedio_altura_cm", "maximo_altura_cm", "minimo_altura_cm"}.issubset(temporada.columns)
assert resultados_sin_deportista == 0

conteo_medallas = df_calculado.filter(F.col("medalla").isin("Gold", "Silver", "Bronze")).count()
conteo_agrupado = medallas_por_equipo.agg(F.sum("total").alias("total")).first()["total"]
assert conteo_medallas == conteo_agrupado, "Los conteos de medallas por equipo no coinciden"

print("VALIDACIONES ESTRUCTURALES: OK")
print("Deportistas:", deportista.count())
print("Resultados integrados:", df_calculado.count())
print("Registros con medalla:", conteo_medallas)
print("Suma de medallas agrupadas:", conteo_agrupado)
print("Eventos sin referencia válida:", resultados_sin_evento)
print("Equipos sin catálogo (id=0):", resultados_sin_equipo)
print("La presencia de datos ausentes está identificada y no provoca pérdida de resultados.")
print("Guardar el notebook con las salidas visibles después de ejecutar Restart & Run All.")

VALIDACIONES ESTRUCTURALES: OK
Deportistas: 135571
Resultados integrados: 271116
Registros con medalla: 39783
Suma de medallas agrupadas: 39783
Eventos sin referencia válida: 9
Equipos sin catálogo (id=0): 200
La presencia de datos ausentes está identificada y no provoca pérdida de resultados.
Guardar el notebook con las salidas visibles después de ejecutar Restart & Run All.


## 9. Video grupal y entrega 

El grupo debe grabar un **video real de 7–10 minutos** y deben intervenir todos los integrantes. Mostrar las secciones del notebook con sus salidas y justificar las decisiones técnicas. Las operaciones, resultados y decisiones técnicas están documentados en las secciones anteriores.

**Antes de entregar:**
1. Ejecutar **Restart & Run All** con VS Code y verificar que ninguna celda falle.
2. Guardar el notebook con todas las salidas visibles y su nombre actual.
3. Grabar el video con el grupo y revisar que dure entre 7 y 10 minutos.
4. Comprimir el notebook y el video en un **ZIP** para la entrega oficial en Canvas. Si el docente necesita ejecutar el notebook desde cero, los archivos originales de `data/` también deberán estar disponibles; consultar o incluirlos en el ZIP.

El notebook publicado se llama `Sumativa_2_grupo7.ipynb`. La guía incluye ejemplos de nombres largos y una regla de hasta 20 caracteres que podrían entrar en conflicto; confirmar la nomenclatura exigida antes de la entrega formal.